# Homework 4 — Evaluation

This notebook documents my solution to the **Module 4 homework of [LLM Zoomcamp](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/cohorts/2026/04-evaluation)** by DataTalksClub.

## What's covered
We generate a ground truth dataset of questions from course lesson pages, build keyword, vector, and hybrid search over the same chunks, and evaluate each method using Hit Rate and MRR — replacing intuition with numbers.

## Stack used

| Component | Tool |
|---|---|
| Document source | `gitsource` (GitHub repo reader) |
| Keyword search | `minsearch.Index` |
| Vector search | `minsearch.VectorSearch` |
| Embeddings | ONNX Runtime (`Xenova/all-MiniLM-L6-v2`) |
| LLM client | Provider-agnostic (Groq, Gemini, OpenRouter) |
| Structured output | `pydantic` |

> API keys are loaded from `.env` via `python-dotenv`. The helper `get_client` returns an OpenAI-compatible client for any supported provider.

## Setup — OpenAI-compatible Client

Most inference providers support the OpenAI-compatible API, so switching providers only requires changing the client configuration.

In [1]:
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
import json
import os
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

ROOT_DIR = Path.cwd().parent
load_dotenv(dotenv_path=ROOT_DIR / ".env")

True

In [2]:
def get_client(provider="groq"):
    config = {
        "groq": {
            "base_url": "https://api.groq.com/openai/v1",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
        "openrouter": {
            "base_url": "https://openrouter.ai/api/v1",
            "api_key": os.getenv("OPENROUTER_API_KEY"),
        },
        "cerebras": {
            "base_url": "https://api.cerebras.ai/v1",
            "api_key": os.getenv("CEREBRAS_API_KEY"),
        },
        "gemini": {
            "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
            "api_key": os.getenv("GEMINI_API_KEY"),
        }
    }
    cfg = config[provider]
    return OpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"])

## Load Documents

We pull the 72 course lesson pages from GitHub, pinned to commit `8c1834d` for reproducibility.

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(f"No. of pages: {len(documents)}")

No. of pages: 72


## Generating Ground Truth

We use an LLM to generate 5 questions per lesson page. The `Questions` Pydantic model enforces structured output. The helper `llm_structured_retry` wraps the LLM call with retry logic.

The instructions ask the LLM to emulate a student, use different wording from the page, and ask about the lesson content.

In [4]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [5]:
class Questions(BaseModel):
    questions: list[str]

In [6]:
from my_evaluation_utils import llm_structured_retry, calc_price, calc_total_price

def generate_ground_truth(doc, client, model, chat_completion):
    user_prompt = json.dumps(doc)
    out, usage = llm_structured_retry(
        client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model=model,
        chat_completion=chat_completion
    )
    results = []
    for q in out.questions:
        results.append({
            "question": q,
            "filename": doc["filename"],
            "content": doc["content"],
        })
    return results, usage

### Q1 — Average input tokens for 3 lesson pages

Generate questions for the first 3 pages and compute the average number of input tokens across the 3 LLM calls.

In [7]:
docs = documents[:3]

ground_truth = []
usages = []

for doc in tqdm(docs):
    records, usage = generate_ground_truth(
        doc,
        get_client("gemini"),
        model="gemini-3.1-flash-lite",
        chat_completion=True
    )
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [8]:
input_tokens = [
    getattr(u, 'input_tokens', getattr(u, 'prompt_tokens', 0))
    for u in usages
]
avg_input_tokens = sum(input_tokens) / len(input_tokens)
print(f"Average Input Tokens: {avg_input_tokens:.0f}")

Average Input Tokens: 1450


## Load Full Ground Truth

The full ground truth of 360 questions (5 per page × 72 pages) is provided as a CSV.

In [9]:
ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = ground_truth.to_dict(orient="records")
print(f"Ground truth records: {len(ground_truth)}")

Ground truth records: 360


## Chunking and Indexing

Each lesson page is split into overlapping 2000-character windows (step=1000). This gives 295 chunks total. We build both a keyword index (`minsearch.Index`) and a vector index (`minsearch.VectorSearch`) over these chunks.

In [10]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks: {len(chunks)}")

Total chunks: 295


### Keyword (Text) Index

In [11]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(chunks)

### Vector Index

Embed each chunk (prepending its filename for context) using ONNX Runtime (`Xenova/all-MiniLM-L6-v2`), producing a 384-dimensional normalized vector per chunk.

In [12]:
from embedder import Embedder

embed = Embedder()
texts = [doc["filename"] + " " + doc["content"] for doc in chunks]

batch_size = 50
X = []
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)
X = np.array(X)
print(f"Embedding matrix shape: {X.shape}")

  0%|          | 0/6 [00:00<?, ?it/s]

Embedding matrix shape: (295, 384)


In [13]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

## Search Functions

In [14]:
def text_search(query, num_results=5):
    return index.search(query, num_results=num_results)


def vector_search(query, num_results=5):
    query_vector = embed.encode(query)
    return vindex.search(query_vector, num_results=num_results)

### Q2 — First result with text search

Take the first question from the ground truth and check which filename ranks #1 in keyword search.

In [15]:
q = ground_truth[0]["question"]
text_result = text_search(q)
print(f"Question: {q}")
print(f"Top result: {text_result[0]['filename']}")

Question: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
Top result: 01-agentic-rag/lessons/03-rag.md


### Q3 — First result with vector search

Same question, but now with semantic (vector) search.

In [16]:
vector_result = vector_search(q)
print(f"Top result: {vector_result[0]['filename']}")

Top result: 01-agentic-rag/lessons/01-intro.md


## Evaluation Metrics

We evaluate search using two standard metrics:
- **Hit Rate**: fraction of questions where the correct page appears in the top-5 results
- **MRR** (Mean Reciprocal Rank): also considers the rank position of the first correct result

In [17]:
def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt += 1
    return cnt / len(relevance)


def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score += 1 / (rank + 1)
                break
    return total_score / len(relevance)


def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])
    relevance = [1 if doc["filename"] == doc_id else 0 for doc in results]
    return relevance


def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)
    return relevance_total


def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total)
    }

### Q4 — Evaluating text search

Evaluate keyword search on the full ground truth. What's the Hit Rate?

In [18]:
evaluate(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

### Q5 — Evaluating vector search

Now evaluate vector search — the part we left for the homework, since the module only evaluated keyword search. What's the MRR?

In [19]:
evaluate(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7277777777777777, 'mrr': 0.5469907407407407}

### Q6 — Tuning hybrid search (RRF k)

Hybrid search combines keyword and vector results using Reciprocal Rank Fusion (RRF). The parameter `k` controls how much the top ranks matter — a smaller `k` sharpens the gap between positions.

Evaluate hybrid search over the full ground truth for `k` values [1, 50, 100, 200] and find which gives the best MRR.

In [25]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [ ]:
# Values of k we want to test
k_values = [1, 50, 100, 200]

# Store MRR for each k
mrr_by_k = {}

# Loop over each k and compute MRR on the full ground‑truth set
for k in k_values:
    # Create a hybrid‑search function that uses the current k
    def hybrid_search(query):
        # Get the two result lists
        vec_res = vector_search(query)
        txt_res = text_search(query)
        # Merge them with RRF using the current k
        return rrf([vec_res, txt_res], k=k, num_results=5)
    
    # Evaluate on the complete ground‑truth dataset
    metrics = evaluate(ground_truth, hybrid_search)
    
    # Save the MRR (and optionally other metrics)
    mrr_by_k[k] = metrics["mrr"]
    print(f"k={k} → MRR = {metrics['mrr']:.4f}")

# -------------------------------------------------
# Pick the best k (smallest k in case of a tie)
# -------------------------------------------------
best_k = min(
    k for k, mrr in mrr_by_k.items() if mrr == max(mrr_by_k.values())
)
print(f"\nBest k: {best_k} (MRR = {mrr_by_k[best_k]:.4f})")

  0%|          | 0/360 [00:00<?, ?it/s]

k=1 → MRR = 0.6192


  0%|          | 0/360 [00:00<?, ?it/s]

k=50 → MRR = 0.6273


  0%|          | 0/360 [00:00<?, ?it/s]

k=100 → MRR = 0.6273


  0%|          | 0/360 [00:00<?, ?it/s]

k=200 → MRR = 0.6273

Best k: 50 (MRR = 0.6273)
